<a href="https://colab.research.google.com/github/mvenyidonny/MyFiles/blob/main/gwp1_prob3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from itertools import combinations
from sklearn.model_selection import train_test_split
from google.colab import drive

# --- Load the Dataset ---
file_path = '/content/drive/My Drive/FE-GWP1_model_selection_1.csv'
# Unmount the drive first if it was previously mounted.
try:
    drive.flush_and_unmount()
    print('All changes made in this colab session should now be visible in your Google Drive.')
except ValueError:
    pass  # Ignore errors if drive was not mounted

# Mount Google Drive.
drive.mount('/content/drive')

# Load phishing dataset
try:
  # data = pd.read_csv("/content/drive/My Drive/FE-GWP1_model_seletion_1.csv")
    df = pd.read_csv(file_path)
    print(f"Dataset '{file_path}' loaded successfully.")
    print("\nDataset Head:")
    print(df.head())
    print("\nDataset Description:")
    print(df.describe())
    print("\nChecking for missing values:")
    print(df.isnull().sum())
except FileNotFoundError:
    print(f"Error: Dataset file not found at '{file_path}'. Please provide the correct path.")
    exit()


print(f"Shape of the initial dataset: {df.shape}")
print(df.head())



All changes made in this colab session should now be visible in your Google Drive.
Mounted at /content/drive
Dataset '/content/drive/My Drive/FE-GWP1_model_selection_1.csv' loaded successfully.

Dataset Head:
          Y        X1        X2        X3        X4        X5
0  3.388410  0.017954 -0.800583 -0.352454  2.187210  1.014887
1  0.287191  0.083057 -0.597947 -0.357639 -1.630284  0.221841
2  3.989645 -0.923437 -1.386575  1.180202  0.632606 -1.576638
3 -2.959602 -0.313775  2.955133 -1.798692 -2.117621  0.159291
4  0.529773  0.388996  1.019611  0.472062  0.590497  0.877048

Dataset Description:
                Y          X1          X2          X3          X4          X5
count  100.000000  100.000000  100.000000  100.000000  100.000000  100.000000
mean     1.257388    0.026830    0.084613   -0.016037    0.122374   -0.201661
std      1.436655    0.481708    0.962145    0.976559    1.076935    1.073358
min     -2.959602   -1.275230   -2.041959   -2.228483   -2.697316   -3.526357
25%    

In [21]:
X_cols = ['X1', 'X2', 'X3', 'X4', 'X5'] # independent variables (features)
y_col = 'Y' # dependent variable (target)


def evaluate_model(X_data, y_data):
    # Ensure X_data is not empty for sm.add_constant
    if X_data.empty:
        # For intercept-only model, provide a constant array
        X_data_const = sm.add_constant(pd.DataFrame({'const': np.ones(len(y_data))}))
    else:
        X_data_const = sm.add_constant(X_data) # Add intercept

    model = sm.OLS(y_data, X_data_const).fit()
    return {
        'adj_r_squared': model.rsquared_adj,
        'aic': model.aic,
        'bic': model.bic,
        'model_summary': model.summary()
    }, model.pvalues, model.params

In [22]:

# --- 1. Backward Elimination Approach ---
print("\n" + "="*50)
print("1. Backward Elimination Approach (Criterion: AIC)")
print("="*50)

current_features_be = list(X_cols)
best_model_be = None
best_aic_be = np.inf
best_bic_be = np.inf
best_adj_r2_be = -np.inf

print(f"Starting with full model: {current_features_be}")


1. Backward Elimination Approach (Criterion: AIC)
Starting with full model: ['X1', 'X2', 'X3', 'X4', 'X5']


In [23]:
while True:
    if not current_features_be: # If current_features list becomes empty
        model_eval, _, _ = evaluate_model(pd.DataFrame(index=df.index), df[y_col]) # Intercept-only model
    else:
        model_eval, p_values, params = evaluate_model(df[current_features_be], df[y_col])

    current_aic = model_eval['aic']
    current_bic = model_eval['bic']
    current_adj_r2 = model_eval['adj_r_squared']

    print(f"\nCurrent Features: {current_features_be if current_features_be else '[Intercept Only]'}")
    print(f"  Adj R-squared: {current_adj_r2:.4f}, AIC: {current_aic:.2f}, BIC: {current_bic:.2f}")

    # Check if this is the best model found so far
    if current_aic < best_aic_be:
        best_aic_be = current_aic
        best_bic_be = current_bic
        best_adj_r2_be = current_adj_r2
        best_model_be = current_features_be.copy()

    if not current_features_be: # Stop if we are at the intercept-only model
        print("  Reached intercept-only model. Stopping backward elimination.")
        break

    # Identify the predictor with the highest p-value (excluding intercept)
    p_values_to_check = p_values.drop('const', errors='ignore') # 'errors=ignore' handles case where 'const' isn't in index

    if p_values_to_check.empty: # This handles cases where only intercept is left, or no predictors to remove
        print("No more predictors to remove based on p-value criteria or only intercept remains.")
        break

    max_p_value = p_values_to_check.max()
    predictor_to_remove = p_values_to_check.idxmax()

    # Create a temporary list of features without the one to be removed
    temp_features = [f for f in current_features_be if f != predictor_to_remove]

    if not temp_features: # If removing this makes it an empty list, consider the intercept-only model
        model_eval_temp, _, _ = evaluate_model(pd.DataFrame(index=df.index), df[y_col])
    else:
        model_eval_temp, _, _ = evaluate_model(df[temp_features], df[y_col])

    aic_after_removal = model_eval_temp['aic']

    # Decision: Remove if AIC improves (decreases)
    if aic_after_removal < current_aic:
        print(f"  Removing '{predictor_to_remove}' (p-value: {max_p_value:.4f}) improved AIC (new AIC: {aic_after_removal:.2f}). Proceeding.")
        current_features_be.remove(predictor_to_remove)
    else:
        print(f"  Removing '{predictor_to_remove}' (p-value: {max_p_value:.4f}) did NOT improve AIC (new AIC: {aic_after_removal:.2f}). Stopping backward elimination.")
        # The best_model_be is already captured in the loop
        break


Current Features: ['X1', 'X2', 'X3', 'X4', 'X5']
  Adj R-squared: 0.6302, AIC: 262.59, BIC: 278.22
  Removing 'X1' (p-value: 0.8805) improved AIC (new AIC: 260.62). Proceeding.

Current Features: ['X2', 'X3', 'X4', 'X5']
  Adj R-squared: 0.6340, AIC: 260.62, BIC: 273.64
  Removing 'X5' (p-value: 0.0206) did NOT improve AIC (new AIC: 264.29). Stopping backward elimination.


In [24]:
print("\nBackward Elimination Final Summary:")
if best_model_be is not None:
    print(f"Best model (based on AIC): {best_model_be}")
    print(f"  Adj R-squared: {best_adj_r2_be:.4f}")
    print(f"  AIC: {best_aic_be:.2f}")
    print(f"  BIC: {best_bic_be:.2f}")
    final_be_model = sm.OLS(df[y_col], sm.add_constant(df[best_model_be]) if best_model_be else sm.add_constant(pd.DataFrame({'const': np.ones(len(df))}))).fit()
    print("\nFull Summary for Best Backward Elimination Model:")
    print(final_be_model.summary())
else:
    print("No best model found by backward elimination (this should not happen).")


Backward Elimination Final Summary:
Best model (based on AIC): ['X2', 'X3', 'X4', 'X5']
  Adj R-squared: 0.6340
  AIC: 260.62
  BIC: 273.64

Full Summary for Best Backward Elimination Model:
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.634
Method:                 Least Squares   F-statistic:                     43.87
Date:                Mon, 16 Jun 2025   Prob (F-statistic):           8.29e-21
Time:                        11:28:05   Log-Likelihood:                -125.31
No. Observations:                 100   AIC:                             260.6
Df Residuals:                      95   BIC:                             273.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 c

In [25]:
# --- 2. Forward Selection Approach ---
print("\n" + "="*50)
print("2. Forward Selection Approach (Criterion: AIC)")
print("="*50)


2. Forward Selection Approach (Criterion: AIC)


In [26]:
remaining_features_fs = list(X_cols)
selected_features_fs = []
best_model_fs = None
best_aic_fs = np.inf
best_bic_fs = np.inf
best_adj_r2_fs = -np.inf

# Evaluate the initial empty model (just intercept)
initial_model_eval, _, _ = evaluate_model(pd.DataFrame(index=df.index), df[y_col])
best_aic_fs = initial_model_eval['aic']
best_bic_fs = initial_model_eval['bic']
best_adj_r2_fs = initial_model_eval['adj_r_squared']
best_model_fs = [] # Represents the intercept-only model

print(f"Starting with empty model (only intercept). Initial AIC: {best_aic_fs:.2f}, BIC: {best_bic_fs:.2f}")

Starting with empty model (only intercept). Initial AIC: 357.25, BIC: 359.85


In [27]:

while remaining_features_fs:
    potential_additions = []

    # Try adding each remaining feature
    for feature in remaining_features_fs:
        test_features = selected_features_fs + [feature]
        model_eval_temp, _, _ = evaluate_model(df[test_features], df[y_col])
        potential_additions.append({
            'feature': feature,
            'aic': model_eval_temp['aic'],
            'bic': model_eval_temp['bic'],
            'adj_r_squared': model_eval_temp['adj_r_squared']
        })

    # Find the feature that offers the best improvement (lowest AIC)
    best_addition = min(potential_additions, key=lambda x: x['aic'])

    if best_addition['aic'] < best_aic_fs:
        selected_features_fs.append(best_addition['feature'])
        remaining_features_fs.remove(best_addition['feature'])

        best_aic_fs = best_addition['aic']
        best_bic_fs = best_addition['bic']
        best_adj_r2_fs = best_addition['adj_r_squared']
        best_model_fs = selected_features_fs.copy()

        print(f"\nAdded '{best_addition['feature']}'. New best model: {selected_features_fs}")
        print(f"  Adj R-squared: {best_adj_r2_fs:.4f}, AIC: {best_aic_fs:.2f}, BIC: {best_bic_fs:.2f}")
    else:
        print("\nAdding any more features does not improve AIC. Stopping forward selection.")
        break


Added 'X4'. New best model: ['X4']
  Adj R-squared: 0.2825, AIC: 325.03, BIC: 330.24

Added 'X3'. New best model: ['X4', 'X3']
  Adj R-squared: 0.4661, AIC: 296.44, BIC: 304.26

Added 'X2'. New best model: ['X4', 'X3', 'X2']
  Adj R-squared: 0.6166, AIC: 264.29, BIC: 274.71

Added 'X5'. New best model: ['X4', 'X3', 'X2', 'X5']
  Adj R-squared: 0.6340, AIC: 260.62, BIC: 273.64

Adding any more features does not improve AIC. Stopping forward selection.


In [28]:
print("\nForward Selection Final Summary:")
if best_model_fs is not None:
    print(f"Best model (based on AIC): {best_model_fs}")
    print(f"  Adj R-squared: {best_adj_r2_fs:.4f}")
    print(f"  AIC: {best_aic_fs:.2f}")
    print(f"  BIC: {best_bic_fs:.2f}")
    final_fs_model = sm.OLS(df[y_col], sm.add_constant(df[best_model_fs]) if best_model_fs else sm.add_constant(pd.DataFrame({'const': np.ones(len(df))}))).fit()
    print("\nFull Summary for Best Forward Selection Model:")
    print(final_fs_model.summary())
else:
    print("No best model found by forward selection (this should not happen).")



Forward Selection Final Summary:
Best model (based on AIC): ['X4', 'X3', 'X2', 'X5']
  Adj R-squared: 0.6340
  AIC: 260.62
  BIC: 273.64

Full Summary for Best Forward Selection Model:
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.634
Method:                 Least Squares   F-statistic:                     43.87
Date:                Mon, 16 Jun 2025   Prob (F-statistic):           8.29e-21
Time:                        11:28:46   Log-Likelihood:                -125.31
No. Observations:                 100   AIC:                             260.6
Df Residuals:                      95   BIC:                             273.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef   

In [29]:
# --- Comparison and Justification ---
print("\n" + "="*50)
print("3. Comparison and Justification of Overall Best Model")
print("="*50)

print("\nSummary of Best Model from Backward Elimination:")
if best_model_be is not None:
    print(f"  Features: {best_model_be}")
    print(f"  Adj R-squared: {best_adj_r2_be:.4f}")
    print(f"  AIC: {best_aic_be:.2f}")
    print(f"  BIC: {best_bic_be:.2f}")
else:
    print("  N/A (Backward Elimination did not find a suitable model).")

print("\nSummary of Best Model from Forward Selection:")
if best_model_fs is not None:
    print(f"  Features: {best_model_fs}")
    print(f"  Adj R-squared: {best_adj_r2_fs:.4f}")
    print(f"  AIC: {best_aic_fs:.2f}")
    print(f"  BIC: {best_bic_fs:.2f}")
else:
    print("  N/A (Forward Selection did not find a suitable model).")


3. Comparison and Justification of Overall Best Model

Summary of Best Model from Backward Elimination:
  Features: ['X2', 'X3', 'X4', 'X5']
  Adj R-squared: 0.6340
  AIC: 260.62
  BIC: 273.64

Summary of Best Model from Forward Selection:
  Features: ['X4', 'X3', 'X2', 'X5']
  Adj R-squared: 0.6340
  AIC: 260.62
  BIC: 273.64


In [30]:
print("\n\nJustification for the Chosen Overall Best Model:")

if best_model_be is not None and best_model_fs is not None:
    if best_aic_be < best_aic_fs:
        chosen_best_model = best_model_be
        chosen_aic = best_aic_be
        chosen_adj_r2 = best_adj_r2_be
        print(f"Comparing the best models from both approaches, the Backward Elimination method yielded a model ({best_model_be}) with a lower AIC ({best_aic_be:.2f}) compared to the Forward Selection model ({best_model_fs}, AIC: {best_aic_fs:.2f}).")
        print("Therefore, the model identified by Backward Elimination is selected as the overall best model based on the AIC criterion.")
    elif best_aic_fs < best_aic_be:
        chosen_best_model = best_model_fs
        chosen_aic = best_aic_fs
        chosen_adj_r2 = best_adj_r2_fs
        print(f"Comparing the best models from both approaches, the Forward Selection method yielded a model ({best_model_fs}) with a lower AIC ({best_aic_fs:.2f}) compared to the Backward Elimination model ({best_model_be}, AIC: {best_aic_be:.2f}).")
        print("Therefore, the model identified by Forward Selection is selected as the overall best model based on the AIC criterion.")
    else:
        chosen_best_model = best_model_be # Arbitrarily pick one if AICs are identical
        chosen_aic = best_aic_be
        chosen_adj_r2 = best_adj_r2_be
        print("Both Backward Elimination and Forward Selection methods yielded models with identical AIC values.")
        print("In such cases, both models are equally good based on AIC. If there's a difference in complexity, the simpler model (fewer features) might be preferred for better interpretability and generalizability (principle of parsimony).")
        print(f"For consistency, we will consider {chosen_best_model} as the chosen best model.")

    print(f"\nFinal Chosen Best Model: {chosen_best_model}")
    print(f"  Adjusted R-squared: {chosen_adj_r2:.4f}")
    print(f"  AIC: {chosen_aic:.2f}")
    print("\nRationale for Decision:")
    print("The choice of AIC as the primary criterion is due to its ability to balance model fit (how well the model explains the variance in 'y') with model complexity (the number of predictors used). A lower AIC indicates a model that is more efficient in explaining the data without overfitting. By employing two distinct stepwise selection methods (Backward Elimination and Forward Selection), we increase the robustness of our selection process. While they might sometimes converge to the same model, their different search paths can sometimes reveal different optimal subsets of features. The model exhibiting the lowest AIC across both methods is ultimately chosen as the most parsimonious and effective model for prediction.")
else:
    print("A definitive overall best model cannot be determined as one or both selection processes did not yield a valid result.")




Justification for the Chosen Overall Best Model:
Both Backward Elimination and Forward Selection methods yielded models with identical AIC values.
In such cases, both models are equally good based on AIC. If there's a difference in complexity, the simpler model (fewer features) might be preferred for better interpretability and generalizability (principle of parsimony).
For consistency, we will consider ['X2', 'X3', 'X4', 'X5'] as the chosen best model.

Final Chosen Best Model: ['X2', 'X3', 'X4', 'X5']
  Adjusted R-squared: 0.6340
  AIC: 260.62

Rationale for Decision:
The choice of AIC as the primary criterion is due to its ability to balance model fit (how well the model explains the variance in 'y') with model complexity (the number of predictors used). A lower AIC indicates a model that is more efficient in explaining the data without overfitting. By employing two distinct stepwise selection methods (Backward Elimination and Forward Selection), we increase the robustness of our 